# Module 2: Taking Advantage of Iceberg Tables

This notebook contains code examples for Module 2 videos.

## Setup

Initialize Spark session for Module 2 examples.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module 2 - Taking Advantage of Iceberg Tables") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Video 1: Moving Existing Data to Iceberg

This video covers three migration strategies:

In [ ]:
spark.sql("""
    CALL polaris.system.snapshot(
        table => 'polaris.taxi.trips_snapshot',
        location => 's3a://warehouse/raw/'
    )
""")

In [ ]:
spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_snapshot
""").show()

spark.sql("""
    SELECT 
        file_path,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        record_count
    FROM polaris.taxi.trips_snapshot.files
""").show(truncate=False)

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_incremental
    USING iceberg
    AS SELECT * FROM polaris.taxi.trips_snapshot WHERE 1=0
""")

spark.sql("""
    CALL polaris.system.add_files(
        table => 'polaris.taxi.trips_incremental',
        source_table => 'parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`'
    )
""")

In [ ]:
spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_incremental
""").show()

### Migration Strategy 3: Reserialization with CTAS

When data isn't in a compatible format (CSV, JSON, etc.) or you need to transform during migration, use CREATE TABLE AS SELECT. This is compute-intensive but provides complete flexibility.

In [ ]:
import tempfile
import os

csv_data = """VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,total_amount
1,2023-06-01 10:15:00,2023-06-01 10:30:00,2,3.5,15.0,20.5
2,2023-06-01 11:20:00,2023-06-01 11:45:00,1,5.2,22.0,28.0
1,2023-06-01 14:30:00,2023-06-01 15:00:00,3,7.8,30.0,38.5
"""

temp_dir = "/tmp/csv_demo"
os.makedirs(temp_dir, exist_ok=True)
csv_path = os.path.join(temp_dir, "sample_trips.csv")

with open(csv_path, 'w') as f:
    f.write(csv_data)

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_from_csv
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS 
    SELECT 
        CAST(VendorID AS INT) as VendorID,
        CAST(tpep_pickup_datetime AS TIMESTAMP) as tpep_pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) as tpep_dropoff_datetime,
        CAST(passenger_count AS INT) as passenger_count,
        CAST(trip_distance AS DOUBLE) as trip_distance,
        CAST(fare_amount AS DOUBLE) as fare_amount,
        CAST(total_amount AS DOUBLE) as total_amount
    FROM csv.`file:///tmp/csv_demo/sample_trips.csv`
""")

In [ ]:
spark.sql("SELECT * FROM polaris.taxi.trips_from_csv").show()

## Video 2: Git-Like Features with Write-Audit-Publish and Branching and Tagging


In [ ]:
# Add a new column to the schema before using MERGE
# In this example, we'll add a correct_fare column (half of the incorrect total_amount)

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN correct_fare DOUBLE
""")

In [ ]:
# Enable Write-Audit-Publish (WAP) and perform MERGE operation
# The changes will be committed but not immediately visible to all readers

spark.conf.set("spark.wap.id", "wap-demo-001")

spark.sql("""
    MERGE INTO polaris.taxi.trips_by_day AS target
    USING (
        SELECT 
            VendorID,
            tpep_pickup_datetime,
            total_amount / 2.0 as correct_fare
        FROM polaris.taxi.trips_by_day
    ) AS source
    ON target.VendorID = source.VendorID 
        AND target.tpep_pickup_datetime = source.tpep_pickup_datetime
    WHEN MATCHED THEN UPDATE SET
        target.correct_fare = source.correct_fare
""")

In [ ]:
# Query the WAP snapshot to audit the changes before publishing
# Normal queries won't see these changes yet

spark.read \
    .option("snapshot-id", "wap-demo-001") \
    .table("polaris.taxi.trips_by_day") \
    .select("total_amount", "correct_fare") \
    .show(10)

In [ ]:
# Option 1: Publish the WAP snapshot to make it visible to all readers
# This promotes the changes from staging to production

spark.sql("""
    CALL polaris.system.publish_changes(
        table => 'polaris.taxi.trips_by_day',
        wap_id => 'wap-demo-001'
    )
""")

In [ ]:
# Option 2 (Alternative): Drop the WAP commit if validation fails
# Iceberg will clean up all data files as if the operation never happened

# spark.sql("""
#     CALL polaris.system.cherrypick_snapshot(
#         table => 'polaris.taxi.trips_by_day',
#         snapshot_id => <wap_snapshot_id>
#     )
# """)

In [ ]:
# Create a branch for experimentation
# This creates an isolated fork of the table history

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day 
    CREATE BRANCH experiment_branch
""")

In [ ]:
# Insert data to the branch only
# Main branch remains unaffected

spark.sql("""
    INSERT INTO polaris.taxi.trips_by_day.branch_experiment_branch
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-10.parquet`
""")

In [ ]:
# Query the branch to verify changes
spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_by_day.branch_experiment_branch
""").show()

In [ ]:
# Merge the branch back to main
# All commits from the branch become part of main's history

spark.sql("""
    CALL polaris.system.fast_forward(
        table => 'polaris.taxi.trips_by_day',
        branch => 'main',
        to => 'experiment_branch'
    )
""")

In [ ]:
# Create a tag to mark an important snapshot
# Tags give snapshots memorable names instead of using snapshot IDs

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    CREATE TAG end_of_quarter
""")

In [ ]:
# Query using the tag
spark.sql("""
    SELECT COUNT(*) as trip_count
    FROM polaris.taxi.trips_by_day.tag_end_of_quarter
""").show()

In [ ]:
# Rollback to a previous snapshot for disaster recovery
# First, let's see the snapshot history

spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation,
        summary
    FROM polaris.taxi.trips_by_day.snapshots
    ORDER BY committed_at DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Rollback to a specific snapshot ID
# Replace <snapshot_id> with the actual ID from the snapshots table

# spark.sql("""
#     CALL polaris.system.rollback_to_snapshot(
#         table => 'polaris.taxi.trips_by_day',
#         snapshot_id => <snapshot_id>
#     )
# """)

## Video 3: Schema Evolution for Iceberg Tables


In [ ]:
# Add a column to the table
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN tip_percentage DOUBLE
""")

In [ ]:
# Rename a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    RENAME COLUMN tip_percentage TO tip_pct
""")

In [ ]:
# Drop a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    DROP COLUMN tip_pct
""")

In [ ]:
# View the table schema to see field IDs
spark.sql("""
    DESCRIBE EXTENDED polaris.taxi.trips_by_day
""").show(truncate=False)

In [ ]:
# Add a column with both initial default and write default
# Initial default: retroactively applies to all existing data
# Write default: applies to future writes when value not provided

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN discount_applied BOOLEAN 
    DEFAULT false
""")

In [ ]:
# Query to verify the default value is applied to existing records
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        total_amount,
        discount_applied
    FROM polaris.taxi.trips_by_day
    LIMIT 10
""").show()

In [ ]:
# Change the write default for future inserts
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ALTER COLUMN discount_applied SET DEFAULT true
""")

## Video 4: Partition Evolution for Iceberg Tables

In [ ]:
# Example: Change partition spec on an existing table
# Start with an unpartitioned table, then add day partitioning

# First, create an unpartitioned table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_evolving
    USING iceberg
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

In [ ]:
# View current partition spec
spark.sql("""
    SELECT * FROM polaris.taxi.trips_evolving.partitions
""").show()

In [ ]:
# Add day partitioning to the table
# Old files remain unchanged; new writes will use day partitioning

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

In [ ]:
# Insert new data - it will use the new partition spec
spark.sql("""
    INSERT INTO polaris.taxi.trips_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

In [ ]:
# View partitions now - mix of unpartitioned and day-partitioned data
spark.sql("""
    SELECT * FROM polaris.taxi.trips_evolving.partitions
""").show()

In [ ]:
# Partition reduction: Remove a partition transform
# This stops future writes from using that partition, but old files remain

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    DROP PARTITION FIELD days(tpep_pickup_datetime)
""")

In [ ]:
# Partition addition: Add multiple partition transforms
# Example: Add both month partitioning and bucket partitioning

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD months(tpep_pickup_datetime)
""")

In [ ]:
# Partition clearing: Remove all partitioning
# Drop all partition fields to return to unpartitioned writes

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    DROP PARTITION FIELD months(tpep_pickup_datetime)
""")

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    DROP PARTITION FIELD bucket(10, PULocationID)
""")

In [ ]:
# Demonstrate automatic predicate transformation
# Query with timestamp filter - Iceberg automatically converts to partition filter

spark.sql("""
    SELECT 
        COUNT(*) as trip_count,
        MIN(tpep_pickup_datetime) as first_trip,
        MAX(tpep_pickup_datetime) as last_trip
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01 00:00:00'
      AND tpep_pickup_datetime < '2023-08-02 00:00:00'
""").show()